# Casting Defect Classification Framework
### Comparative Analysis of a Custom CNN and Transfer Learning Models (MobileNetV2, ResNet50) with Explainable AI (Grad-CAM + SHAP)

This notebook trains and compares three deep-learning models on the **Real-life Industrial Casting Product** dataset (submersible-pump impeller images), a binary classification task: `def_front` (defective) vs `ok_front` (OK).

It produces **every** output required by the class activity:
- Dataset description, model-architecture tables, evaluation metrics, confusion matrices, training curves
- Grad-CAM visualizations (correct + misclassified)
- SHAP visualizations
- A final comparison table

All **figures are saved as PDF** and all **tables as CSV** into `/kaggle/working/outputs`, then the whole folder is **zipped for direct download**.

---
## Before you run (Kaggle settings)
1. **Add the dataset**: *Add Input* -> search "Real-life Industrial Dataset of Casting Product" (by ravirajsinh45) and attach it.
2. **Turn ON the GPU**: *Settings -> Accelerator -> GPU* (training is much faster).
3. **Turn ON Internet**: *Settings -> Internet -> On* (required to download the ImageNet pre-trained weights for MobileNetV2 and ResNet50). If Internet is off, the notebook still runs but those two models start from random weights and become trainable.
4. **Run All**.
5. When finished, download the zip from the **Output** tab: `casting_defect_outputs.zip`.


In [ ]:
# === Imports & reproducibility ===
import os, sys, time, json, glob, random, shutil, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU available:", bool(tf.config.list_physical_devices("GPU")))


In [ ]:
# === Configuration, output dirs, helpers, dataset auto-detection ===

# ---- Hyperparameters ----
IMG_SIZE        = (224, 224)
BATCH_SIZE      = 32
EPOCHS          = 15
VAL_SPLIT       = 0.20
QUICK_RUN       = False     # set True for a fast smoke-test (2 epochs)
if QUICK_RUN:
    EPOCHS = 2

GRADCAM_N       = 4         # images per Grad-CAM figure
SHAP_N_IMAGES   = 2         # test images explained with SHAP
SHAP_MAX_EVALS  = 300       # SHAP evaluations per image

# ---- Output directories ----
WORKING_DIR = "/kaggle/working"
OUTPUT_DIR  = os.path.join(WORKING_DIR, "outputs")
FIG_DIR     = os.path.join(OUTPUT_DIR, "figures")
TAB_DIR     = os.path.join(OUTPUT_DIR, "tables")
for d in (OUTPUT_DIR, FIG_DIR, TAB_DIR):
    os.makedirs(d, exist_ok=True)

def fig_path(name):
    return os.path.join(FIG_DIR, name)

def tab_path(name):
    return os.path.join(TAB_DIR, name)

# Colormap for Grad-CAM overlays (no cv2 dependency)
JET = plt.get_cmap("jet")

def save_message_figure(path, title, message):
    "Save a placeholder figure when a real one cannot be produced."
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.axis("off")
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.text(0.5, 0.5, message, ha="center", va="center", wrap=True, fontsize=11)
    fig.savefig(path, bbox_inches="tight")
    plt.show()
    plt.close(fig)

# ---- Dataset auto-detection ----
# The user-supplied path plus the standard Kaggle slug path are both tried,
# and we also scan all of /kaggle/input. We look for a directory that holds
# BOTH 'train' and 'test' subfolders, each containing >= 2 class subdirs.
CANDIDATE_ROOTS = [
    "/kaggle/input/datasets/ravirajsinh45/real-life-industrial-dataset-of-casting-product",
    "/kaggle/input/real-life-industrial-dataset-of-casting-product",
]
CANDIDATE_ROOTS += sorted(glob.glob("/kaggle/input/*"))

def _has_classes(d):
    if not os.path.isdir(d):
        return False
    subs = [s for s in os.listdir(d) if os.path.isdir(os.path.join(d, s))]
    return len(subs) >= 2

def find_dataset_dirs():
    "Return (train_dir, test_dir) by walking candidate roots."
    seen = set()
    roots = []
    for r in CANDIDATE_ROOTS:
        if r and r not in seen:
            seen.add(r); roots.append(r)
    for root in roots:
        if not os.path.isdir(root):
            continue
        for cur, dirs, _ in os.walk(root):
            lower = {x.lower(): x for x in dirs}
            if "train" in lower and "test" in lower:
                tr = os.path.join(cur, lower["train"])
                te = os.path.join(cur, lower["test"])
                if _has_classes(tr) and _has_classes(te):
                    return tr, te
    return None, None

TRAIN_DIR, TEST_DIR = find_dataset_dirs()
if TRAIN_DIR is None:
    raise FileNotFoundError(
        "Could not locate train/test folders under /kaggle/input. "
        "Make sure the casting-product dataset is attached as an Input."
    )
print("TRAIN_DIR:", TRAIN_DIR)
print("TEST_DIR :", TEST_DIR)

CLASS_NAMES = sorted(s for s in os.listdir(TRAIN_DIR)
                     if os.path.isdir(os.path.join(TRAIN_DIR, s)))
NUM_CLASSES = len(CLASS_NAMES)
print("Classes:", CLASS_NAMES)


## 2. Dataset Description

In [ ]:
# === Dataset description tables ===
def count_images(folder):
    counts = {}
    for c in CLASS_NAMES:
        cdir = os.path.join(folder, c)
        n = 0
        if os.path.isdir(cdir):
            n = len([f for f in os.listdir(cdir)
                     if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))])
        counts[c] = n
    return counts

train_counts = count_images(TRAIN_DIR)
test_counts  = count_images(TEST_DIR)

rows = []
for c in CLASS_NAMES:
    rows.append({
        "Class": c,
        "Train images": train_counts[c],
        "Test images":  test_counts[c],
        "Total": train_counts[c] + test_counts[c],
    })
dist_df = pd.DataFrame(rows)
dist_df.loc["Total"] = ["TOTAL",
                        dist_df["Train images"].sum(),
                        dist_df["Test images"].sum(),
                        dist_df["Total"].sum()]
dist_df.to_csv(tab_path("dataset_metadata.csv"), index=False)

desc_df = pd.DataFrame([
    {"Property": "Dataset name", "Value": "Real-life Industrial Casting Product (ravirajsinh45)"},
    {"Property": "Task",          "Value": "Binary image classification"},
    {"Property": "Number of classes", "Value": NUM_CLASSES},
    {"Property": "Class names",   "Value": ", ".join(CLASS_NAMES)},
    {"Property": "Image size used", "Value": f"{IMG_SIZE[0]}x{IMG_SIZE[1]} RGB"},
    {"Property": "Total train images", "Value": int(dist_df.loc['Total','Train images'])},
    {"Property": "Total test images",  "Value": int(dist_df.loc['Total','Test images'])},
])
desc_df.to_csv(tab_path("dataset_description.csv"), index=False)

print(desc_df.to_string(index=False))
print()
print(dist_df.to_string(index=False))


In [ ]:
# === Build tf.data datasets ===
# Raw images in [0,255]; preprocessing is baked INTO each model so that the
# same raw pipeline feeds every model and Grad-CAM/SHAP see interpretable pixels.

train_ds_raw = keras.utils.image_dataset_from_directory(
    TRAIN_DIR, validation_split=VAL_SPLIT, subset="training", seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, color_mode="rgb",
    label_mode="int", class_names=CLASS_NAMES, shuffle=True)

val_ds_raw = keras.utils.image_dataset_from_directory(
    TRAIN_DIR, validation_split=VAL_SPLIT, subset="validation", seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, color_mode="rgb",
    label_mode="int", class_names=CLASS_NAMES, shuffle=True)

test_ds_raw = keras.utils.image_dataset_from_directory(
    TEST_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE, color_mode="rgb",
    label_mode="int", class_names=CLASS_NAMES, shuffle=False)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds_raw.prefetch(AUTOTUNE)
val_ds   = val_ds_raw.cache().prefetch(AUTOTUNE)
test_ds  = test_ds_raw.cache().prefetch(AUTOTUNE)

print("Batches -> train:", tf.data.experimental.cardinality(train_ds).numpy(),
      "val:", tf.data.experimental.cardinality(val_ds).numpy(),
      "test:", tf.data.experimental.cardinality(test_ds).numpy())


In [ ]:
# === Figure 1: Dataset samples (a few images per class) ===
per_class = GRADCAM_N
collected = {c: [] for c in CLASS_NAMES}
for imgs, labels in train_ds_raw.unbatch():
    c = CLASS_NAMES[int(labels.numpy())]
    if len(collected[c]) < per_class:
        collected[c].append(imgs.numpy().astype("uint8"))
    if all(len(v) >= per_class for v in collected.values()):
        break

rows_n = NUM_CLASSES
cols_n = per_class
fig, axes = plt.subplots(rows_n, cols_n, figsize=(cols_n * 2.4, rows_n * 2.4))
axes = np.array(axes).reshape(rows_n, cols_n)
for r, c in enumerate(CLASS_NAMES):
    for k in range(cols_n):
        ax = axes[r, k]
        ax.axis("off")
        if k < len(collected[c]):
            ax.imshow(collected[c][k])
        if k == 0:
            ax.set_title(c, loc="left", fontsize=12, fontweight="bold")
fig.suptitle("Figure 1: Dataset Samples", fontsize=14, fontweight="bold")
fig.tight_layout()
fig.savefig(fig_path("Figure1_dataset_samples.pdf"), bbox_inches="tight")
plt.show()
plt.close(fig)


In [ ]:
# === Figure 2: Model design workflow diagram ===
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(7, 9))
ax.set_xlim(0, 10); ax.set_ylim(0, 12); ax.axis("off")

def box(y, text, color):
    b = FancyBboxPatch((2, y), 6, 1.1, boxstyle="round,pad=0.1",
                       linewidth=1.5, edgecolor="black", facecolor=color)
    ax.add_patch(b)
    ax.text(5, y + 0.55, text, ha="center", va="center", fontsize=11, fontweight="bold")

def arrow(y0, y1):
    ax.add_patch(FancyArrowPatch((5, y0), (5, y1), arrowstyle="-|>",
                 mutation_scale=18, linewidth=1.5, color="black"))

box(10.3, "Input Image (224x224 RGB)", "#cfe8ff")
arrow(10.3, 9.5)
box(8.4, "Custom CNN / MobileNetV2 / ResNet50", "#d8f5d8")
arrow(8.4, 7.6)
box(6.5, "Prediction (def_front / ok_front)", "#fff2cc")
arrow(6.5, 5.7)
box(4.6, "Evaluation (Acc, P, R, F1, Conf. Matrix)", "#ffe0cc")
arrow(4.6, 3.8)
box(2.7, "Grad-CAM + SHAP Explanations", "#f0d0ff")

fig.suptitle("Figure 2: Model Design Workflow", fontsize=14, fontweight="bold")
fig.savefig(fig_path("Figure2_workflow.pdf"), bbox_inches="tight")
plt.show()
plt.close(fig)


## 3. Methodology — Model Architectures

Three models share an identical raw input pipeline; preprocessing is embedded in each model:
- **Custom CNN** — built from scratch with `Rescaling(1/255)` + light augmentation.
- **MobileNetV2** — ImageNet weights (frozen), input scaled to `[-1, 1]`.
- **ResNet50** — ImageNet weights (frozen), Caffe-style preprocessing.

If Internet is off and ImageNet weights cannot be downloaded, the base networks fall back to random initialization and are made trainable.

In [ ]:
# === Build the three models ===
from tensorflow.keras.applications import MobileNetV2, ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess

def augmentation_block(x):
    x = layers.RandomFlip("horizontal", seed=SEED)(x)
    x = layers.RandomRotation(0.05, seed=SEED)(x)
    x = layers.RandomZoom(0.10, seed=SEED)(x)
    return x

def load_base(ctor):
    "Try ImageNet weights; fall back to random weights if download fails."
    try:
        base = ctor(include_top=False, weights="imagenet",
                    input_shape=IMG_SIZE + (3,))
        base.trainable = False
        return base, True
    except Exception as e:
        print("  [warn] could not load ImageNet weights (", type(e).__name__,
              ") -> using random weights, base trainable.")
        base = ctor(include_top=False, weights=None,
                    input_shape=IMG_SIZE + (3,))
        base.trainable = True
        return base, False

def build_custom_cnn():
    inp = keras.Input(shape=IMG_SIZE + (3,))
    x = augmentation_block(inp)
    x = layers.Rescaling(1.0 / 255)(x)
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    feat = layers.Conv2D(128, 3, padding="same", activation="relu",
                         name="last_conv")(x)
    x = layers.MaxPooling2D()(feat)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation="relu")(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    model = keras.Model(inp, out, name="Custom_CNN")
    return model, feat

def build_transfer(ctor, scale_fn, name):
    inp = keras.Input(shape=IMG_SIZE + (3,))
    x = augmentation_block(inp)
    x = scale_fn(x)
    base, pretrained = load_base(ctor)
    feat = base(x, training=False)
    g = layers.GlobalAveragePooling2D()(feat)
    g = layers.Dropout(0.3)(g)
    out = layers.Dense(NUM_CLASSES, activation="softmax")(g)
    model = keras.Model(inp, out, name=name)
    return model, feat

def mobilenet_scale(x):
    return layers.Rescaling(1.0 / 127.5, offset=-1.0)(x)

def resnet_scale(x):
    return layers.Lambda(resnet_preprocess, name="resnet_preprocess")(x)

cnn_model,  cnn_feat  = build_custom_cnn()
mob_model,  mob_feat  = build_transfer(MobileNetV2, mobilenet_scale, "MobileNetV2_TL")
res_model,  res_feat  = build_transfer(ResNet50,    resnet_scale,    "ResNet50_TL")

MODELS = {
    "Custom CNN":  {"model": cnn_model, "feat": cnn_feat},
    "MobileNetV2": {"model": mob_model, "feat": mob_feat},
    "ResNet50":    {"model": res_model, "feat": res_feat},
}

opt_kwargs = dict(loss="sparse_categorical_crossentropy", metrics=["accuracy"])
for name, d in MODELS.items():
    d["model"].compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), **opt_kwargs)
    d["model"].build((None,) + IMG_SIZE + (3,))
    print(f"{name:12s} trainable params built.")


In [ ]:
# === Model architecture summary table ===
def _num(weights):
    total = 0
    for w in weights:
        s = 1
        for dim in w.shape:
            s *= int(dim)
        total += s
    return total

arch_rows = []
for name, d in MODELS.items():
    m = d["model"]
    trainable = _num(m.trainable_weights)
    non_trainable = _num(m.non_trainable_weights)
    arch_rows.append({
        "Model": name,
        "Total params": trainable + non_trainable,
        "Trainable params": trainable,
        "Non-trainable params": non_trainable,
        "Layers": len(m.layers),
    })
    # full text summary -> .txt
    with open(tab_path(f"architecture_{name.replace(' ', '_')}.txt"), "w") as fh:
        m.summary(print_fn=lambda line: fh.write(line + "\n"))

arch_df = pd.DataFrame(arch_rows)
arch_df.to_csv(tab_path("model_architecture_summary.csv"), index=False)
print(arch_df.to_string(index=False))


## 4. Training

In [ ]:
# === Train all three models ===
models_info = {}
for name, d in MODELS.items():
    print(f"\n===== Training {name} =====")
    cb = keras.callbacks.EarlyStopping(monitor="val_loss", patience=4,
                                       restore_best_weights=True)
    t0 = time.time()
    hist = d["model"].fit(train_ds, validation_data=val_ds,
                          epochs=EPOCHS, callbacks=[cb], verbose=2)
    elapsed = time.time() - t0
    models_info[name] = {
        "history": hist.history,
        "train_time_s": elapsed,
        "epochs_run": len(hist.history["loss"]),
    }
    print(f"{name}: {elapsed:.1f}s over {models_info[name]['epochs_run']} epochs.")


In [ ]:
# === Figure 3: Accuracy and loss curves ===
fig, axes = plt.subplots(NUM_CLASSES if False else 3, 2, figsize=(12, 12))
for i, (name, info) in enumerate(models_info.items()):
    h = info["history"]
    ax_a, ax_l = axes[i, 0], axes[i, 1]
    ax_a.plot(h["accuracy"], label="train")
    ax_a.plot(h["val_accuracy"], label="val")
    ax_a.set_title(f"{name} — Accuracy"); ax_a.set_xlabel("epoch")
    ax_a.set_ylabel("accuracy"); ax_a.legend(); ax_a.grid(alpha=0.3)
    ax_l.plot(h["loss"], label="train")
    ax_l.plot(h["val_loss"], label="val")
    ax_l.set_title(f"{name} — Loss"); ax_l.set_xlabel("epoch")
    ax_l.set_ylabel("loss"); ax_l.legend(); ax_l.grid(alpha=0.3)
fig.suptitle("Figure 3: Training & Validation Curves", fontsize=14, fontweight="bold")
fig.tight_layout()
fig.savefig(fig_path("Figure3_training_curves.pdf"), bbox_inches="tight")
plt.show()
plt.close(fig)


## 4.1 Model Evaluation

In [ ]:
# === Materialize the test set once, predict, compute metrics ===
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)

X_test_list, y_test_list = [], []
for imgs, labels in test_ds_raw:   # unshuffled, consistent order
    X_test_list.append(imgs.numpy())
    y_test_list.append(labels.numpy())
X_test = np.concatenate(X_test_list, axis=0).astype("float32")
y_test = np.concatenate(y_test_list, axis=0).astype("int64")
print("Test tensor:", X_test.shape)

eval_rows = []
preds_store = {}
for name, d in MODELS.items():
    probs = d["model"].predict(X_test, batch_size=BATCH_SIZE, verbose=0)
    y_pred = probs.argmax(axis=1)
    preds_store[name] = {"probs": probs, "y_pred": y_pred}

    acc = accuracy_score(y_test, y_pred)
    p_w, r_w, f_w, _ = precision_recall_fscore_support(
        y_test, y_pred, average="weighted", zero_division=0)
    p_m, r_m, f_m, _ = precision_recall_fscore_support(
        y_test, y_pred, average="macro", zero_division=0)
    eval_rows.append({
        "Model": name, "Accuracy": acc,
        "Precision (weighted)": p_w, "Recall (weighted)": r_w, "F1 (weighted)": f_w,
        "Precision (macro)": p_m, "Recall (macro)": r_m, "F1 (macro)": f_m,
        "Train time (s)": models_info[name]["train_time_s"],
    })

    rep = classification_report(y_test, y_pred, target_names=CLASS_NAMES,
                                output_dict=True, zero_division=0)
    pd.DataFrame(rep).transpose().to_csv(
        tab_path(f"classification_report_{name.replace(' ', '_')}.csv"))

eval_df = pd.DataFrame(eval_rows)
eval_df.to_csv(tab_path("evaluation_results.csv"), index=False)
print(eval_df.to_string(index=False))


In [ ]:
# === Figure 4: Confusion matrix comparison ===
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (name, store) in zip(axes, preds_store.items()):
    cm = confusion_matrix(y_test, store["y_pred"])
    pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
        tab_path(f"confusion_matrix_{name.replace(' ', '_')}.csv"))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(name, fontweight="bold")
    ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
    ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    thr = cm.max() / 2.0 if cm.max() else 0
    for r in range(NUM_CLASSES):
        for c in range(NUM_CLASSES):
            ax.text(c, r, int(cm[r, c]), ha="center", va="center",
                    color="white" if cm[r, c] > thr else "black", fontsize=12)
fig.suptitle("Figure 4: Confusion Matrices", fontsize=14, fontweight="bold")
fig.tight_layout()
fig.savefig(fig_path("Figure4_confusion_matrices.pdf"), bbox_inches="tight")
plt.show()
plt.close(fig)


## 4.3 Grad-CAM Explainability

In [ ]:
# === Grad-CAM helpers ===
from numpy.random import default_rng
rng = default_rng(SEED)

grad_models = {}
for name, d in MODELS.items():
    grad_models[name] = keras.Model(d["model"].inputs,
                                    [d["feat"], d["model"].output])

def make_gradcam_heatmap(img_array, name, pred_index=None):
    gm = grad_models[name]
    img_array = tf.convert_to_tensor(img_array[None].astype("float32"))
    with tf.GradientTape() as tape:
        conv_out, preds = gm(img_array, training=False)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]
    grads = tape.gradient(class_channel, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_out = conv_out[0]
    heatmap = conv_out @ pooled[..., None]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def overlay_heatmap(img_uint8, heatmap, alpha=0.45):
    hm = tf.image.resize(heatmap[..., None], IMG_SIZE).numpy().squeeze()
    hm_color = JET(hm)[..., :3]
    overlay = (1 - alpha) * (img_uint8 / 255.0) + alpha * hm_color
    return np.clip(overlay, 0, 1)


In [ ]:
# === Figure 5: Grad-CAM on images correctly classified by ALL models ===
correct_all = np.where(np.all(
    [preds_store[n]["y_pred"] == y_test for n in MODELS], axis=0))[0]

if len(correct_all) == 0:
    save_message_figure(fig_path("Figure5_gradcam_correct.pdf"),
                        "Figure 5: Grad-CAM (correct predictions)",
                        "No image was correctly classified by all three models.")
else:
    sel = rng.choice(correct_all, size=min(GRADCAM_N, len(correct_all)),
                     replace=False)
    n = len(sel); ncol = len(MODELS) + 1
    fig, axes = plt.subplots(n, ncol, figsize=(ncol * 3, n * 3))
    axes = np.array(axes).reshape(n, ncol)
    for r, idx in enumerate(sel):
        img = X_test[idx].astype("uint8")
        axes[r, 0].imshow(img); axes[r, 0].axis("off")
        axes[r, 0].set_title(f"Original\n(true: {CLASS_NAMES[y_test[idx]]})", fontsize=9)
        for k, name in enumerate(MODELS, start=1):
            hm = make_gradcam_heatmap(X_test[idx], name)
            axes[r, k].imshow(overlay_heatmap(img, hm)); axes[r, k].axis("off")
            axes[r, k].set_title(name, fontsize=9)
    fig.suptitle("Figure 5: Grad-CAM — Correct Predictions", fontsize=14, fontweight="bold")
    fig.tight_layout()
    fig.savefig(fig_path("Figure5_gradcam_correct.pdf"), bbox_inches="tight")
    plt.show()
    plt.close(fig)


In [ ]:
# === Figure 6: Grad-CAM on misclassified images (very important) ===
wrong_union = np.where(np.any(
    [preds_store[n]["y_pred"] != y_test for n in MODELS], axis=0))[0]

if len(wrong_union) == 0:
    save_message_figure(fig_path("Figure6_gradcam_misclassified.pdf"),
                        "Figure 6: Grad-CAM (misclassifications)",
                        "No misclassifications occurred — every model classified every test image correctly.")
else:
    sel = rng.choice(wrong_union, size=min(GRADCAM_N, len(wrong_union)),
                     replace=False)
    n = len(sel); ncol = len(MODELS) + 1
    fig, axes = plt.subplots(n, ncol, figsize=(ncol * 3, n * 3.2))
    axes = np.array(axes).reshape(n, ncol)
    for r, idx in enumerate(sel):
        img = X_test[idx].astype("uint8")
        axes[r, 0].imshow(img); axes[r, 0].axis("off")
        axes[r, 0].set_title(f"Original\n(true: {CLASS_NAMES[y_test[idx]]})", fontsize=9)
        for k, name in enumerate(MODELS, start=1):
            pred = preds_store[name]["y_pred"][idx]
            hm = make_gradcam_heatmap(X_test[idx], name, pred_index=int(pred))
            axes[r, k].imshow(overlay_heatmap(img, hm)); axes[r, k].axis("off")
            mark = "OK" if pred == y_test[idx] else "X"
            axes[r, k].set_title(f"{name}\npred: {CLASS_NAMES[pred]} [{mark}]", fontsize=9)
    fig.suptitle("Figure 6: Grad-CAM — Misclassifications", fontsize=14, fontweight="bold")
    fig.tight_layout()
    fig.savefig(fig_path("Figure6_gradcam_misclassified.pdf"), bbox_inches="tight")
    plt.show()
    plt.close(fig)


## 4.4 SHAP Explainability

In [ ]:
# === Figure 7: SHAP explanations (one figure per model) ===
try:
    import shap
except Exception:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "shap"], check=False)
    import shap

shap_idx = rng.choice(len(X_test), size=min(SHAP_N_IMAGES, len(X_test)),
                      replace=False)
shap_images = (X_test[shap_idx] / 255.0).astype("float32")  # SHAP works in [0,1]

masker = shap.maskers.Image("blur(64,64)", shap_images[0].shape)

for name, d in MODELS.items():
    model = d["model"]
    def predict_fn(x, _m=model):
        return _m.predict((x * 255.0).astype("float32"), verbose=0)
    out_pdf = fig_path(f"Figure7_SHAP_{name.replace(' ', '_')}.pdf")
    try:
        explainer = shap.Explainer(predict_fn, masker, output_names=CLASS_NAMES)
        sv = explainer(shap_images, max_evals=SHAP_MAX_EVALS,
                       batch_size=BATCH_SIZE)
        shap.image_plot(sv, show=False)
        fig = plt.gcf()
        fig.suptitle(f"Figure 7: SHAP — {name}", fontsize=13, fontweight="bold")
        fig.savefig(out_pdf, bbox_inches="tight")
        plt.show()
        plt.close(fig)
    except Exception as e:
        save_message_figure(out_pdf, f"Figure 7: SHAP — {name}",
                            f"SHAP could not be computed: {type(e).__name__}: {e}")


## 5. Final Comparison

In [ ]:
# === Final comparison table ===
best_f1   = eval_df.loc[eval_df["F1 (weighted)"].idxmax(), "Model"]
fastest   = eval_df.loc[eval_df["Train time (s)"].idxmin(), "Model"]
smallest  = arch_df.loc[arch_df["Total params"].idxmin(), "Model"]

def finding(name):
    bits = []
    if name == best_f1:  bits.append("best F1")
    if name == fastest:  bits.append("fastest to train")
    if name == smallest: bits.append("fewest parameters")
    return "; ".join(bits) if bits else "baseline"

final_rows = []
for name in MODELS:
    e = eval_df[eval_df["Model"] == name].iloc[0]
    a = arch_df[arch_df["Model"] == name].iloc[0]
    final_rows.append({
        "Model": name,
        "Accuracy": round(float(e["Accuracy"]), 4),
        "F1-Score": round(float(e["F1 (weighted)"]), 4),
        "Training Time (s)": round(float(e["Train time (s)"]), 1),
        "Parameters": int(a["Total params"]),
        "Grad-CAM Quality": "",        # student fills: High / Medium / Low
        "SHAP Interpretability": "",   # student fills: High / Medium / Low
        "Overall Finding": finding(name),
    })
final_df = pd.DataFrame(final_rows)
final_df.to_csv(tab_path("final_comparison.csv"), index=False)
print(final_df.to_string(index=False))
print("\nNote: 'Grad-CAM Quality' and 'SHAP Interpretability' are left blank")
print("for you to fill in qualitatively (High / Medium / Low) after inspecting")
print("Figures 5-7.")


In [ ]:
# === Zip the entire outputs directory for direct download ===
zip_base = os.path.join(WORKING_DIR, "casting_defect_outputs")
zip_path = shutil.make_archive(zip_base, "zip", OUTPUT_DIR)
print("Created:", zip_path, f"({os.path.getsize(zip_path)/1e6:.2f} MB)")

print("\nContents of outputs/:")
for root, _, files in os.walk(OUTPUT_DIR):
    for f in sorted(files):
        rel = os.path.relpath(os.path.join(root, f), OUTPUT_DIR)
        print("  ", rel)

try:
    from IPython.display import FileLink, display
    display(FileLink(zip_path))
except Exception:
    pass


## 6. Discussion & Conclusion — Mapping Outputs to Research Questions

- **RQ1 (Custom CNN accuracy):** see `evaluation_results.csv` and Figure 3/4 for the Custom CNN row.
- **RQ2 (Does transfer learning improve over the custom CNN?):** compare the Accuracy / F1 of MobileNetV2 and ResNet50 against the Custom CNN in `final_comparison.csv`.
- **RQ3 (Best performance/cost trade-off?):** weigh F1-Score against Training Time and Parameters in `final_comparison.csv`.
- **RQ4 (Where does each model look on correct predictions?):** Figure 5 (Grad-CAM correct).
- **RQ5 (Different attention patterns CNN vs transfer?):** compare columns within Figure 5.
- **RQ6 (SHAP positive/negative evidence?):** Figure 7 (red = pushes toward the class, blue = against).
- **RQ7 (Can explainability diagnose misclassifications?):** Figure 6 (Grad-CAM on errors).

### Filling the qualitative columns
After inspecting Figures 5-7, rate each model's **Grad-CAM Quality** and **SHAP Interpretability** as **High / Medium / Low** (or 1-5) in `final_comparison.csv`:
- *High* = the highlighted regions clearly fall on the impeller defect / relevant structure.
- *Low* = attention is scattered on the background or irrelevant areas.

All figures (PDF) and tables (CSV) are inside `casting_defect_outputs.zip` in the **Output** tab.